# 📊 View — Top Categorias

Validação da view `vw_top_categorias` antes de mover para o Streamlit.

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.append('..')

#formata todos os números float com 2 casas decimais na exibição do Jupyter.
pd.set_option('display.float_format', '{:.2f}'.format)

pedidos    = pd.read_csv("../dados/pedidos_limpo.csv", parse_dates=[
    'order_purchase_timestamp',
    'order_approved_at',
    'order_delivered_carrier_date',
    'order_delivered_customer_date',
    'order_estimated_delivery_date'
])

clientes = pd.read_csv("../dados/clientes_limpo.csv")
itens      = pd.read_csv("../dados/itens_limpo.csv", parse_dates=['shipping_limit_date'])
produtos   = pd.read_csv("../dados/produtos_limpo.csv")

print("Dados carregados!")


Dados carregados!


## 🧪 Testando o código antes de criar a view

In [3]:
itens.head(2)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93


In [4]:
produtos.head(1)

,product_id,product_category_name,product_name_lenght,product_description_lenght,product_photos_qty,product_weight_g,product_length_cm,product_height_cm,product_width_cm
0,1e9e8ef04dbcff4541ed26657ea517e5,perfumaria,40.00,287.00,1,225.00,16.00,10.00,14.00


In [2]:

df = (itens
          .merge(produtos[['product_id', 'product_category_name']], on='product_id', how='left')
          .merge(pedidos[['order_id', 'order_status', 'order_purchase_timestamp', 'customer_id']], on='order_id', how='left')
          .merge(clientes[['customer_id', 'customer_state']], on='customer_id', how='left')) # <-- Novo merge

# Filtrando só pedidos entregues
df = df[df['order_status'] == 'delivered']

df.head(2)

,order_id,order_item_id,product_id,seller_id,shipping_limit_date,price,freight_value,product_category_name,order_status,order_purchase_timestamp,customer_id,customer_state
0,00010242fe8c5a6d1ba2dd792cb16214,1,4244733e06e7ecb4970a6e2683c13e61,48436dade18ac8b2bce089ec2a041202,2017-09-19 09:45:35,58.90,13.29,cool_stuff,delivered,2017-09-13 08:59:02,3ce436f183e68e07877b285a838db11a,RJ
1,00018f77f2f0320c557190d7a144bdd3,1,e5f2d52b802189ee658865ca93d83a8f,dd7ddc04e1b6c2c614352b383efe2d36,2017-05-03 11:05:13,239.90,19.93,pet_shop,delivered,2017-04-26 10:53:06,f6dd3ec061db4e3987629fe6b26e5cce,SP


In [8]:

df['ano']             = df['order_purchase_timestamp'].dt.year
df['data_mes']        = df['order_purchase_timestamp'].dt.to_period('M').astype(str)
df['receita_produto'] = df['price']
df['receita_frete']   = df['freight_value']
df['receita_item']    = df['price'] + df['freight_value']

top_categorias = (df.groupby(['customer_state', 'product_category_name', 'ano', 'data_mes'])
                   .agg(
                       total_pedidos    = ('order_id',        'nunique'),
                       total_itens      = ('order_item_id',   'count'),
                       total_vendedores = ('seller_id',       'nunique'),
                       receita_produtos = ('receita_produto', 'sum'),
                       receita_frete    = ('receita_frete',   'sum'),
                       receita_total    = ('receita_item',    'sum'),
                   )
                   .reset_index()
                   .sort_values('receita_total', ascending=False))

top_categorias['receita_total']    = top_categorias['receita_total'].round(2)
top_categorias['receita_produtos'] = top_categorias['receita_produtos'].round(2)
top_categorias['receita_frete']    = top_categorias['receita_frete'].round(2)
top_categorias['ticket_medio']     = (top_categorias['receita_total'] / top_categorias['total_pedidos']).round(2)
top_categorias['preco_medio_item'] = (top_categorias['receita_produtos'] / top_categorias['total_itens']).round(2)
top_categorias['valor_medio_item_com_frete'] = (top_categorias['receita_total'] / top_categorias['total_itens']).round(2)

top_categorias.head(10)

,customer_state,product_category_name,ano,data_mes,total_pedidos,total_itens,total_vendedores,receita_produtos,receita_frete,receita_total,ticket_medio,preco_medio_item,valor_medio_item_com_frete
10546,SP,beleza_saude,2018,2018-08,370,409,120,49676.60,5711.26,55387.86,149.70,121.46,135.42
11381,SP,relogios_presentes,2018,2018-05,231,248,25,47480.02,2582.61,50062.63,216.72,191.45,201.87
10543,SP,beleza_saude,2018,2018-05,342,384,100,42537.50,5058.91,47596.41,139.17,110.77,123.95
10544,SP,beleza_saude,2018,2018-06,366,421,122,41407.70,5900.56,47308.26,129.26,98.36,112.37
11050,SP,informatica_acessorios,2018,2018-02,309,375,58,39849.49,5344.27,45193.76,146.26,106.27,120.52
10586,SP,cama_mesa_banho,2018,2018-06,350,417,71,37740.78,7154.87,44895.65,128.27,90.51,107.66
10585,SP,cama_mesa_banho,2018,2018-05,327,411,62,38908.24,5980.76,44889.00,137.28,94.67,109.22
10542,SP,beleza_saude,2018,2018-04,293,333,102,39665.18,4648.49,44313.67,151.24,119.11,133.07
10588,SP,cama_mesa_banho,2018,2018-08,320,381,74,37305.79,6875.53,44181.32,138.07,97.92,115.96
10579,SP,cama_mesa_banho,2017,2017-11,332,393,46,36656.44,5713.82,42370.26,127.62,93.27,107.81


### IMPORTANDO VIEW

In [6]:
from views.vw_top_categorias import get_top_categorias

df_categorias = get_top_categorias(pedidos, itens, produtos, clientes)
df_categorias.head()

,customer_state,product_category_name,ano,data_mes,total_pedidos,total_itens,total_vendedores,receita_produtos,receita_frete,receita_total,ticket_medio,preco_medio_item
10546,SP,beleza_saude,2018,2018-08,370,409,120,49676.60,5711.26,55387.86,149.70,121.46
11381,SP,relogios_presentes,2018,2018-05,231,248,25,47480.02,2582.61,50062.63,216.72,191.45
10543,SP,beleza_saude,2018,2018-05,342,384,100,42537.50,5058.91,47596.41,139.17,110.77
10544,SP,beleza_saude,2018,2018-06,366,421,122,41407.70,5900.56,47308.26,129.26,98.36
11050,SP,informatica_acessorios,2018,2018-02,309,375,58,39849.49,5344.27,45193.76,146.26,106.27


In [7]:
top_categorias.head(10)

,customer_state,product_category_name,ano,data_mes,total_pedidos,total_itens,total_vendedores,receita_produtos,receita_frete,receita_total,ticket_medio,preco_medio_item
10546,SP,beleza_saude,2018,2018-08,370,409,120,49676.60,5711.26,55387.86,149.70,121.46
11381,SP,relogios_presentes,2018,2018-05,231,248,25,47480.02,2582.61,50062.63,216.72,191.45
10543,SP,beleza_saude,2018,2018-05,342,384,100,42537.50,5058.91,47596.41,139.17,110.77
10544,SP,beleza_saude,2018,2018-06,366,421,122,41407.70,5900.56,47308.26,129.26,98.36
11050,SP,informatica_acessorios,2018,2018-02,309,375,58,39849.49,5344.27,45193.76,146.26,106.27
10586,SP,cama_mesa_banho,2018,2018-06,350,417,71,37740.78,7154.87,44895.65,128.27,90.51
10585,SP,cama_mesa_banho,2018,2018-05,327,411,62,38908.24,5980.76,44889.00,137.28,94.67
10542,SP,beleza_saude,2018,2018-04,293,333,102,39665.18,4648.49,44313.67,151.24,119.11
10588,SP,cama_mesa_banho,2018,2018-08,320,381,74,37305.79,6875.53,44181.32,138.07,97.92
10579,SP,cama_mesa_banho,2017,2017-11,332,393,46,36656.44,5713.82,42370.26,127.62,93.27
